# Comparación por sujeto de LightGBM mixto y V6 con HMM

### Qué modelo va mejor

Pruebas guardadas de cinco rondas por persona: LightGBM **5/5** y V6 **1/5**.

V6 usa la CRNN con ventanas de 15 segmentos, LightGBM con 192 características temporales y decodificación HMM por noche (fase B, una pasada).

**Resultado provisional.** En las rondas compartidas (1), LightGBM mixto obtiene el mayor F1 macro, con una ventaja de 0,0445.

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |


El reporte completo se encuentra en **5. Resultados fuera de muestra**, al final del notebook.

In [1]:
from pathlib import Path
import hashlib
import json
import os

import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.dvc/config').is_file())
notebook = ROOT / 'Notebooks/experiments/comparacion_justa_lightgbm_v6.ipynb'
sources = [c['source'] for c in json.loads(notebook.read_text('utf-8'))['cells'] if c['id'] in {'cv-01','cv-03','cv-05','cv-07','cv-09','cv-10'}]
fingerprint = hashlib.sha256(json.dumps(sources, sort_keys=True).encode())
for puntero in sorted((ROOT / 'datos').glob('*.dvc')):
    fingerprint.update(puntero.read_bytes())
for source in [ROOT / 'scripts/compare_cassette_hybrid.py', *sorted((ROOT / 'packages/sleep-staging/src/sleep_staging').rglob('*.py'))]:
    fingerprint.update(source.read_bytes())
CACHE = Path(os.environ.get('SLEEP_CV_DIR', ROOT / '.sleep-cv')) / fingerprint.hexdigest()[:16]

def mostrar_resultados(carpeta):
    """Lee las predicciones de prueba y las mediciones guardadas, sin entrenar modelos."""
    nombres = {'dart': 'LightGBM mixto', 'v6': 'CRNN + LightGBM 192D + HMM'}
    etapas = ['W', 'N1', 'N2', 'N3', 'REM']
    claves = ['record_id', 'subject', 'epoch_index', 'stage', 'fold']
    texto = []
    def tabla(columnas, filas):
        texto.extend(['| ' + ' | '.join(columnas) + ' |',
                      '| ' + ' | '.join(['---'] * len(columnas)) + ' |'])
        texto.extend('| ' + ' | '.join(map(str, fila)) + ' |' for fila in filas)
        texto.append('')
    def titulo(t):
        texto.extend(['### ' + t, ''])
    def numero(x, dec=4):
        return f'{x:.{dec}f}'.replace('.', ',')
    def cohorte(serie):
        c = serie.str[:2].map({'ST':'Telemetry', 'SC':'Cassette'})
        assert c.notna().all()
        return c
    def puntuar(datos):
        y, p = datos.stage, datos.prediction
        return [f1_score(y, p, labels=etapas, average='macro', zero_division=0),
                accuracy_score(y, p), cohen_kappa_score(y, p)]
    def fila(etiqueta, datos):
        f1, acc, k = puntuar(datos)
        return [etiqueta, datos.subject.nunique(), datos.record_id.nunique(), len(datos),
                numero(f1), numero(100*acc, 2)+' %', numero(k)]
    columnas = ['Modelo','Personas','Noches','Segmentos de 30 s','F1 macro','Exactitud','Kappa']
    pred = {m:{} for m in nombres}
    for m in nombres:
        for f in range(1,6):
            p = carpeta / f'fold-{f}-{m}.parquet'
            if p.is_file():
                d = pd.read_parquet(p)
                assert d.fold.eq(f).all() and d.stage.isin(etapas).all() and d.prediction.isin(etapas).all()
                assert not d.duplicated(['record_id','epoch_index']).any()
                d['cohorte'] = cohorte(d.record_id)
                pred[m][f] = d
    comunes = sorted(pred['dart'].keys() & pred['v6'].keys())
    for f in comunes:
        a, b = [pred[m][f][claves].sort_values(claves).reset_index(drop=True) for m in nombres]
        pd.testing.assert_frame_equal(a, b)
    titulo('Qué modelo va mejor')
    texto.extend([f"Pruebas guardadas de cinco rondas por persona: LightGBM **{len(pred['dart'])}/5** y V6 **{len(pred['v6'])}/5**.", '',
                  'V6 usa la CRNN con ventanas de 15 segmentos, LightGBM con 192 características temporales y decodificación HMM por noche (fase B, una pasada).', ''])
    if comunes:
        par = {m:pd.concat([pred[m][f] for f in comunes]) for m in nombres}
        diferencia = puntuar(par['dart'])[0] - puntuar(par['v6'])[0]
        ganador = nombres['dart' if diferencia > 0 else 'v6']
        alcance = 'Resultado de las cinco rondas' if len(comunes)==5 else 'Resultado provisional'
        conclusion = (f'{ganador} obtiene el mayor F1 macro, con una ventaja de {numero(abs(diferencia))}.'
                      if diferencia else 'Los dos modelos obtienen el mismo F1 macro.')
        texto.extend([f"**{alcance}.** En las rondas compartidas ({', '.join(map(str,comunes))}), {conclusion}", ''])
        tabla(columnas, [fila(nombres[m], par[m]) for m in nombres])
        titulo('Rendimiento por ronda y cohorte de prueba')
        tabla(['Ronda','Cohorte',*columnas], [
            [f, c, *fila(nombres[m], grupo)] for f in comunes for m in nombres
            for c, grupo in [('Ambas',pred[m][f]), *pred[m][f].groupby('cohorte')]])
        titulo('Matrices de confusión de las rondas compartidas')
        texto.extend(['Cada fila indica la etapa real y cada columna la predicción. Los valores cuentan segmentos de 30 segundos.', ''])
        for m in nombres:
            matriz = pd.crosstab(par[m].stage, par[m].prediction).reindex(index=etapas, columns=etapas, fill_value=0)
            assert matriz.to_numpy().sum() == len(par[m])
            texto.extend(['**'+nombres[m]+'**',''])
            tabla(['Real / predicha',*etapas], [[e,*matriz.loc[e].tolist()] for e in etapas])
    completos = [m for m in nombres if len(pred[m]) == 5]
    if completos:
        titulo('Rendimiento global sobre las cinco rondas')
        tabla(columnas, [fila(nombres[m], pd.concat(pred[m].values())) for m in completos])
        datos = pd.concat(pred[completos[0]].values())
        assert datos.groupby('subject').fold.nunique().eq(1).all() and not datos.duplicated(['record_id','epoch_index']).any()
        titulo('Datos de entrenamiento y prueba')
        texto.extend(['Los dos modelos comparten esta partición. Cada noche se cuenta una vez en prueba y cuatro veces en entrenamiento a lo largo de la validación cruzada. Una persona conserva todas sus noches en la misma ronda de prueba.', ''])
        tabla(['Cohorte','Personas disponibles','Noches disponibles','Segmentos etiquetados'],
              [[c,g.subject.nunique(),g.record_id.nunique(),len(g)] for c,g in datos.groupby('cohorte')])
        filas, ids = [], []
        for f in range(1,6):
            for c,g in datos.groupby('cohorte'):
                tr, te = g[g.fold.ne(f)], g[g.fold.eq(f)]
                assert set(tr.subject).isdisjoint(te.subject)
                filas.append([f,c,tr.subject.nunique(),tr.record_id.nunique(),len(tr),
                             te.subject.nunique(),te.record_id.nunique(),len(te)])
                ids.append([f,c,', '.join(sorted(te.record_id.unique()))])
        tabla(['Ronda','Cohorte','Entrenamiento · personas','Noches','Segmentos',
               'Prueba · personas','Noches','Segmentos'], filas)
        titulo('Qué noches quedaron en prueba')
        texto.extend(['Para reconstruir el entrenamiento de una ronda, toma las noches de las otras cuatro rondas de esta tabla.', ''])
        tabla(['Ronda','Cohorte','Identificadores de los registros de prueba'], ids)
    titulo('Tiempo de entrenamiento de la primera ronda')
    carpeta_t = carpeta / 'tiempos-ronda-1'
    tiempos = carpeta_t / 'mediciones.json'
    crnn = carpeta_t / 'epocas-crnn.json'
    info = json.loads(tiempos.read_text()) if tiempos.exists() else {}
    ajustes = []
    if 'ajuste_dart_s' in info:
        ajustes.append(['LightGBM mixto','Ajuste repetido sobre las características del mismo entrenamiento (CPU)',numero(info['ajuste_dart_s'],1)])
    if crnn.exists():
        epocas = json.loads(crnn.read_text())
        ajustes.append(['V6 · extractor CRNN',f'Suma de {len(epocas)} épocas del registro original de Keras (RTX 3080)',numero(sum(epocas),1)])
    archivo_vectores=carpeta_t/'extraccion-entrenamiento.json'
    if archivo_vectores.exists():
        vectores=json.loads(archivo_vectores.read_text())
        ajustes.append(['V6 · extracción de vectores',f"Estimación para {vectores['noches']} noches de entrenamiento; intervalos entre guardados consecutivos de la ejecución original",numero(vectores['segundos'],1)])
    if 'ajuste_cabezal_hmm_s' in info:
        ajustes.append(['V6 · LightGBM + HMM','Reajuste sobre los vectores guardados de la primera ronda (CPU)',numero(info['ajuste_cabezal_hmm_s'],1)])
        if crnn.exists() and archivo_vectores.exists():
            total = sum(epocas) + vectores['segundos'] + info['ajuste_cabezal_hmm_s']
            ajustes.append(['V6 · total aproximado','Suma de las tres etapas anteriores sobre el entrenamiento',numero(total,1)])
    if ajustes:
        for ajuste in ajustes:
            segundos = round(float(ajuste[-1].replace(',', '.')))
            horas, resto = divmod(segundos, 3600)
            minutos, segundos = divmod(resto, 60)
            ajuste.append(f'{horas:d} h {minutos:02d} min {segundos:02d} s')
        tabla(['Modelo / componente','Qué se cronometró','Segundos','Duración'], ajustes)
    texto.extend(['Las mediciones corresponden a la primera ronda, con las entradas preparadas. El tiempo de ajuste de LightGBM comienza con las características tabulares disponibles. En V6 comienza con las señales filtradas disponibles. El total V6 combina registros originales con un nuevo cronometraje del clasificador final.', ''])
    titulo('Tiempo de inferencia de una noche completa desde el EDF')
    if info.get('inferencia'):
        texto.extend([f"Una pasada por modelo sobre la primera noche de prueba de cada cohorte, ordenada por identificador. Modelos en memoria, tras calentamiento. CRNN en {info.get('dispositivo_crnn','GPU')} y LightGBM en CPU. El total incluye lectura, preprocesamiento y predicción de todos los segmentos del EDF. El F1 usa solo los segmentos con etiqueta válida.", ''])
        tabla(['Cohorte','Noche','Modelo','Segmentos EDF','Preparación (s)','Predicción (s)','Total (s)','Coincidencia con predicción guardada'],
              [[r['cohorte'],r['registro'],nombres[r['modelo']],r['segmentos_edf'],
                numero(r['preparacion_s'],2),numero(r['prediccion_s'],2),numero(r['total_s'],2),
                numero(100*r['coincidencia'],2)+' %'] for r in info['inferencia']])
    else:
        texto.extend(['Pendiente de medición desde EDF.', ''])
    control = carpeta_t / 'validacion-st7051.json'
    if control.is_file():
        v = json.loads(control.read_text())
        texto.extend([f"Al repetir LightGBM en {v['registro']} desde el EDF, {v['diferencias_edf']} de {v['n']} segmentos etiquetados cambiaron de predicción. Con el mismo clasificador y las características tabulares guardadas se reprodujeron las {v['n']} predicciones. Las tablas de rendimiento usan las predicciones originales de prueba.", ''])
    return '\n'.join(texto)

reporte = mostrar_resultados(CACHE)
display(Markdown(reporte.split('### Rendimiento por ronda')[0]))

### Qué modelo va mejor

Pruebas guardadas de cinco rondas por persona: LightGBM **5/5** y V6 **1/5**.

V6 usa la CRNN con ventanas de 15 segmentos, LightGBM con 192 características temporales y decodificación HMM por noche (fase B, una pasada).

**Resultado provisional.** En las rondas compartidas (1), LightGBM mixto obtiene el mayor F1 macro, con una ventaja de 0,0445.

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |



## Protocolo de comparación

**Pregunta:** ¿qué modelo clasifica mejor una noche completa de una persona ajena al entrenamiento? Usamos las mismas noches y etiquetas en cinco particiones por persona, equilibradas por cohorte. Cada sujeto aparece una sola vez en prueba. La métrica principal es F1 macro de las cinco etapas sobre todas las predicciones fuera de muestra. También se informa por sujeto, cohorte y fold. El recorte de vigilia se aplica solo en una evaluación secundaria.

Se comparan el LightGBM DART de Jhoan y la fase de V6 con mejor F1 histórico (CRNN de 15 épocas, LightGBM de 192 variables y HMM). Conservamos las configuraciones elegidas en experimentos previos. Cada fold parte de parámetros sin entrenar. Es validación cruzada de desarrollo sobre datos ya explorados.

Desde la raíz, prepara el entorno del proyecto y ejecuta `dvc pull`. El entrenamiento completo de V6 requiere bastante tiempo; se recomienda GPU. Las predicciones y puntos de reanudación se guardan fuera del dataset DVC. La carpeta aparece al ejecutar la configuración.

In [1]:
import gc
import hashlib
import json
import os
import sys
import time
import zipfile
from pathlib import Path
from tempfile import gettempdir

import mne
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.dvc/config').is_file())
sys.path.insert(0, str(ROOT / 'packages/sleep-staging/src'))
sys.path.insert(0, str(ROOT))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import VALID_STAGES, discover_sleep_cassette_records, discover_sleep_telemetry_records, read_epoch_labels
from sleep_staging.training import load_or_build_supervised_dataset
from scripts.compare_cassette_hybrid import contextual_embeddings, decode_night

SEED = 42
FOLDS = 5
EPOCHS = 20
BATCH = 16  # Ajustado a la memoria de una GPU de 10 GB.
THREADS = 6
STAGES = list(VALID_STAGES)
DATA = ROOT / 'datos'
spec = json.loads((DATA / 'experimentos/lightgbm_st_sc/experimento.json').read_text('utf-8'))
reference = DATA / 'referencias/hibrido_v6/modelo_crnn_maestro_optimo_v7.keras'
with zipfile.ZipFile(reference) as archive:
    architecture = json.loads(archive.read('config.json'))
architecture.pop('compile_config', None)  # Solo la topología. Pesos y optimizador nuevos por fold.

devices = tf.config.list_physical_devices('GPU')
for gpu in devices:
    tf.config.experimental.set_memory_growth(gpu, True)
print('GPU:', devices)
tf.keras.utils.set_random_seed(SEED)
records_st = discover_sleep_telemetry_records(DATA / 'sleep-telemetry')
records_sc = discover_sleep_cassette_records(DATA / 'sleep-cassette')
records = records_st + records_sc
lookup = {r.record_id: r for r in records}
assert len(lookup) == len(records)
cohort_st = {r.record_id for r in records_st}
subjects = pd.DataFrame([
    (r.subject_id, 'Telemetry' if r.record_id in cohort_st else 'Cassette')
    for r in records
], columns=['subject', 'cohort']).drop_duplicates().sort_values(['cohort', 'subject']).reset_index(drop=True)
assert subjects.subject.is_unique
subjects['fold'] = 0
for fold, (_, test) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(subjects, subjects.cohort), 1):
    subjects.loc[test, 'fold'] = fold

CACHE.mkdir(parents=True, exist_ok=True)
print('Reanudación:', CACHE)
display(subjects.pivot_table(index='fold', columns='cohort', values='subject', aggfunc='count'))
display(subjects.groupby('fold').subject.agg(list).to_frame('sujetos reservados'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Reanudación: /mnt/e/PROJECTS/Maestria Inteligencia Artificial/masterIADesarollo/micro-proyecto-grupo-7/.sleep-cv/8f83a22b1d3a65d3


cohort,Cassette,Telemetry
fold,,
1,3,4
2,3,4
3,2,5
4,2,5
5,2,4


,sujetos reservados
fold,
1,"[SC410, SC434, SC435, S05, S14, S21, S22]"
2,"[SC426, SC452, SC472, S07, S15, S19, S24]"
3,"[SC457, SC482, S04, S09, S10, S12, S20]"
4,"[SC407, SC417, S02, S06, S11, S13, S17]"
5,"[SC406, SC441, S01, S08, S16, S18]"


## 1. Población y características comunes

La tabla maestra conserva una fila por (registro, número de época) con una etiqueta válida. Todas las variables tabulares se calculan por noche, como en el notebook original. Telemetry usa dos EEG, EOG y EMG. En Cassette se calculan las características de los tres canales a 100 Hz y se dejan ausentes las del EMG de alta frecuencia. La columna de recorte solo interviene al calcular la métrica secundaria.

In [2]:
def tabular_data(record_list, profile, cache_dir):
    pipeline = PreprocessingPipeline(PreprocessingConfig(**spec[profile]))
    ds = load_or_build_supervised_dataset(record_list, pipeline, cache_dir, verbose=False)
    frame = ds.features.astype('float32').reindex(columns=spec['feature_columns'])
    meta = ds.metadata[['record_id', 'subject_id', 'epoch_index']].copy()
    meta['stage'] = ds.labels.to_numpy()
    return frame, meta

parts = [tabular_data(records_st, 'preprocessing_telemetry', DATA / 'processed/supervised')]
for record in records_sc:
    parts.append(tabular_data([record], 'preprocessing_cassette', Path(gettempdir()) / 'sleep-edfx-sc-features-v1'))
X = pd.concat([p[0] for p in parts], ignore_index=True)
meta = pd.concat([p[1] for p in parts], ignore_index=True).rename(columns={'subject_id': 'subject'})
del parts
meta = meta.join(subjects.set_index('subject'), on='subject', validate='many_to_one')
assert meta.fold.notna().all() and len(X) == len(meta)
assert not meta.duplicated(['record_id', 'epoch_index']).any() and set(meta.stage) == set(STAGES)
meta['trim'] = True
for _, night in meta.groupby('record_id', sort=False):
    sleep = night.loc[night.stage.ne('W'), 'epoch_index']
    if len(sleep):
        meta.loc[night.index, 'trim'] = night.epoch_index.between(sleep.min()-60, sleep.max()+60)
for fold in range(1, FOLDS + 1):
    held = meta.fold.eq(fold)
    assert set(meta.loc[held, 'subject']).isdisjoint(meta.loc[~held, 'subject'])
display(meta.groupby(['cohort', 'fold']).agg(sujetos=('subject', 'nunique'), noches=('record_id', 'nunique'), épocas=('stage', 'size')))

/home/sebas/.local/share/sleep-edfx-cv/tmp/ipykernel_491/2503506403.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  X = pd.concat([p[0] for p in parts], ignore_index=True)


sujetos  noches  épocas
cohort    fold                         
Cassette  1           3       3    8039
          2           3       3    8240
          3           2       2    5594
          4           2       2    5490
          5           2       2    5558
Telemetry 1           4       8    7854
          2           4       8    7523
          3           5      10    9815
          4           5      10    9499
          5           4       8    8000

## 2. LightGBM mixto

Ajustamos el modelo DART con los hiperparámetros fijados en el notebook de Jhoan, incluyendo sus pesos de clase. Ambos modelos reciben exactamente los índices de entrenamiento y prueba de la tabla anterior.

In [3]:
def save_predictions(path, test_rows, predictions):
    result = meta.loc[test_rows, ['record_id', 'subject', 'epoch_index', 'stage', 'fold']].copy()
    result['prediction'] = predictions
    assert result.prediction.isin(STAGES).all()
    temp = path.with_suffix('.tmp.parquet')
    result.to_parquet(temp, index=False)
    temp.replace(path)

for fold in range(1, FOLDS + 1):
    target = CACHE / f'fold-{fold}-dart.parquet'
    if target.exists():
        continue
    train = meta.fold.ne(fold)
    test = meta.index[~train]
    params = dict(spec['parameters'], n_jobs=THREADS)
    dart = LGBMClassifier(**params)
    start = time.perf_counter()
    dart.fit(X.loc[train], meta.loc[train, 'stage'])
    dart.booster_.save_model(str(CACHE / f'fold-{fold}-dart.txt'))
    save_predictions(target, test, dart.predict(X.loc[test]))
    print(f'LightGBM · fold {fold} · {time.perf_counter() - start:.0f} s', flush=True)
    del dart
del X
gc.collect()

0

## 3. Señales del híbrido

El híbrido conserva las cuatro señales, el filtro 0,3–35 Hz y el z-score por canal dentro de cada época. Los mapas de memoria permiten construir ventanas de 15 épocas sin duplicar en RAM todas las ventanas de todas las noches. Las ventanas respetan el tiempo original del EDF y nunca cruzan registros. Solo la época central necesita etiqueta. El hipnograma permite comprobar la alineación y no se usa como entrada.

In [4]:
channels = spec['preprocessing_telemetry']['channels']
record_names = list(lookup)
record_number = {rid: i for i, rid in enumerate(record_names)}
night_number = meta.record_id.map(record_number).to_numpy(dtype=int)
epoch_number = meta.epoch_index.to_numpy(dtype=int)
y_num = pd.Categorical(meta.stage, categories=STAGES).codes.astype('int32')

def prepare_night(rid):
    record = lookup[rid]
    path = CACHE / f'{rid}-signal.npy'
    if not path.exists():
        raw = mne.io.read_raw_edf(record.psg_path, preload=True, verbose='ERROR')
        if raw.info['sfreq'] != 100:
            raise ValueError(f'Frecuencia inesperada en {rid}')
        present = [ch for ch in channels if ch in raw.ch_names]
        raw.pick(present).filter(0.3, 35.0, verbose=False)
        n = int(raw.n_times // 3000)
        tmp = path.with_suffix('.tmp.npy')
        buf = np.lib.format.open_memmap(tmp, mode='w+', dtype='float32', shape=(n, 3000, len(channels)))
        for j, ch in enumerate(channels):
            if ch in present:
                v = (raw.get_data(picks=[ch], stop=n * 3000).reshape(n, 3000) * 1e6)
                buf[:, :, j] = (v - v.mean(1, keepdims=True)) / (v.std(1, keepdims=True) + 1e-8)
            else:
                buf[:, :, j] = 0
        buf.flush()
        del buf, raw
        tmp.replace(path)
    data = np.load(path, mmap_mode='r')
    labels = read_epoch_labels(record.hypnogram_path, len(data))
    night = meta.loc[meta.record_id.eq(rid)]
    np.testing.assert_array_equal(np.flatnonzero(np.isin(labels, STAGES)), night.epoch_index)
    np.testing.assert_array_equal(labels[night.epoch_index.to_numpy()], night.stage)
    return data

signals = [prepare_night(rid) for rid in record_names]
print(f'{len(signals)} noches alineadas; {len(meta):,} épocas supervisadas.')

56 noches alineadas; 75,612 épocas supervisadas.


## 4. CRNN, clasificador contextual y HMM

Se instancia la arquitectura de V6 desde su configuración, con inicialización aleatoria por fold. Usamos 20 épocas de entrenamiento, focal loss (γ = 1,5), ponderación por clase calculada solo en entrenamiento y el calendario del extractor original. El lote físico es de 16 ventanas; acumulamos gradientes para aproximar el lote nominal de 128 del experimento original. Las estadísticas de Batch Normalization se calculan en cada lote físico.

El segundo LightGBM se ajusta sobre los embeddings de entrenamiento del extractor del fold, siguiendo el procedimiento del modelo final del compañero. Las transiciones del HMM y sus probabilidades iniciales se estiman solo con las noches de entrenamiento. En prueba se decodifica cada noche completa antes de seleccionar las épocas puntuables. La versión con HMM obtuvo el mejor F1 macro histórico; se usa una sola pasada.

In [5]:
class NightBatches(tf.keras.utils.PyDataset):
    def __init__(self, rows, seed):
        super().__init__(workers=2, use_multiprocessing=False, max_queue_size=3)
        self.rows = np.asarray(rows).copy()
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()

    def __len__(self):
        return (len(self.rows) + BATCH - 1) // BATCH

    def on_epoch_end(self):
        self.rng.shuffle(self.rows)

    def __getitem__(self, batch):
        rows = self.rows[batch * BATCH:(batch + 1) * BATCH]
        x = np.empty((len(rows), 15, 3000, 4), dtype='float32')
        for night in np.unique(night_number[rows]):
            mask = night_number[rows] == night
            centers = epoch_number[rows[mask], None]
            steps = centers + np.arange(-7, 8)
            src = signals[night]
            x[mask] = src[np.where((steps >= 0) & (steps < len(src)), steps, centers)]
        return {'seq_eeg': x[..., :2], 'seq_eog': x[..., 2:3], 'seq_emg': x[..., 3:]}, tf.keras.utils.to_categorical(y_num[rows], len(STAGES)).astype('float32')

def focal(y, p):
    p = tf.clip_by_value(tf.cast(p, tf.float32), 1e-7, 1 - 1e-7)
    return -tf.reduce_sum(y * (1 - p)**1.5 * tf.math.log(p), axis=-1)

def learning_rate(epoch, _):
    return (0.001, 0.0005, 0.00025, 0.000125)[sum(epoch >= t for t in (6, 12, 16))]

def train_encoder(train, fold, folder):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED + fold)
    net = tf.keras.models.model_from_json(json.dumps(architecture))
    weights = folder / 'encoder.weights.h5'
    if weights.exists():
        net.load_weights(weights)  # Entrenado exclusivamente con los sujetos de este fold.
    else:
        net.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, gradient_accumulation_steps=128 // BATCH),
                    loss=focal, jit_compile=False)
        counts = np.bincount(y_num[train], minlength=len(STAGES))
        assert (counts > 0).all()
        class_weight = dict(enumerate(np.sqrt(len(train) / (len(STAGES) * counts))))
        callbacks = [
            tf.keras.callbacks.LearningRateScheduler(learning_rate),
            tf.keras.callbacks.BackupAndRestore(str(folder / 'epoch-backup'), double_checkpoint=True, delete_checkpoint=False),
            tf.keras.callbacks.CSVLogger(str(folder / 'training.csv'), append=True),
        ]
        net.fit(NightBatches(train, SEED + fold), epochs=EPOCHS, class_weight=class_weight,
                callbacks=callbacks, shuffle=False, verbose=2)
        tmp = folder / 'pending.weights.h5'
        net.save_weights(tmp)
        tmp.replace(weights)
    return tf.keras.Model(net.inputs, net.get_layer('embeddings_latentes_64d').output)

def fit_hmm(train_meta):
    counts = np.ones((len(STAGES), len(STAGES)), dtype=float)
    prior = np.ones(len(STAGES))
    for _, night in train_meta.groupby('record_id'):
        night = night.sort_values('epoch_index')
        codes = pd.Categorical(night.stage, categories=STAGES).codes
        adjacent = np.diff(night.epoch_index.to_numpy()) == 1
        np.add.at(counts, (codes[:-1][adjacent], codes[1:][adjacent]), 1)
        np.add.at(prior, codes[np.r_[True, ~adjacent]], 1)
    return counts / counts.sum(axis=1, keepdims=True), prior / prior.sum()

# Parámetros del clasificador final V6 de 192 variables. El modelo histórico
# serializado usa bagging_freq=0 (subsample_freq en la interfaz de sklearn).
hybrid_params = dict(n_estimators=600, learning_rate=0.0644405, num_leaves=62,
                     max_depth=7, min_child_samples=130, subsample=0.743569,
                     subsample_freq=0, colsample_bytree=0.583576, reg_alpha=0.376046,
                     reg_lambda=1.4647, class_weight='balanced', random_state=42,
                     n_jobs=THREADS, verbosity=-1)

In [ ]:
for fold in range(1, FOLDS + 1):
    target = CACHE / f'fold-{fold}-v6.parquet'
    if target.exists():
        continue
    folder = CACHE / f'v6-{fold}'
    folder.mkdir(exist_ok=True)
    train = meta.index[meta.fold.ne(fold)].to_numpy()
    test = meta.index[meta.fold.eq(fold)].to_numpy()
    print(f'V6 · fold {fold} · {len(train)} épocas de ajuste', flush=True)
    encoder = train_encoder(train, fold, folder)
    embeddings = {}
    for rid in record_names:
        cache_file = folder / f'{rid}-192.npy'
        if not cache_file.exists():
            a = contextual_embeddings(encoder, signals[record_number[rid]], batch_size=BATCH)
            tmp = cache_file.with_suffix('.tmp.npy')
            np.save(tmp, a)
            tmp.replace(cache_file)
        embeddings[rid] = np.load(cache_file, mmap_mode='r')
    features = np.empty((len(train), 192), dtype='float32')
    train_nights = night_number[train]
    for j in np.unique(train_nights):
        mask = train_nights == j
        features[mask] = embeddings[record_names[j]][epoch_number[train[mask]]]
    head = LGBMClassifier(**hybrid_params).fit(features, y_num[train])
    assert np.array_equal(head.classes_, np.arange(len(STAGES)))
    transition, prior = fit_hmm(meta.loc[train])
    head.booster_.save_model(str(folder / 'clasificador-192.txt'))
    np.savez(folder / 'hmm.npz', transition=transition, prior=prior)
    pred = pd.Series(index=test, dtype=object)
    for rid, night in meta.loc[test].groupby('record_id'):
        probabilities = head.predict_proba(embeddings[rid])
        path = decode_night(probabilities, transition, prior)
        pred.loc[night.index] = np.array(STAGES)[path[night.epoch_index.to_numpy()]]
    save_predictions(target, test, pred.loc[test].to_numpy())
    del encoder, embeddings, features, head
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
def medir_tiempos_ronda_1():
    """Cronometra el ajuste tabular y la inferencia desde EDF en dos noches reservadas."""
    from lightgbm import Booster

    carpeta = CACHE / 'tiempos-ronda-1'
    carpeta.mkdir(exist_ok=True)
    destino = carpeta / 'mediciones.json'
    info = json.loads(destino.read_text()) if destino.exists() else {'inferencia': []}
    def guardar():
        temporal = destino.with_suffix('.tmp')
        temporal.write_text(json.dumps(info, indent=2), encoding='utf-8')
        temporal.replace(destino)

    train = meta.index[meta.fold.ne(1)].to_numpy()
    test = meta.loc[meta.fold.eq(1)]
    rutas = {k: carpeta / f'{k}.txt' for k in ('dart', 'v6')}
    if not rutas['dart'].exists() or 'ajuste_dart_s' not in info:
        if 'X' in globals():
            tabla = X
        else:
            partes = [tabular_data(records_st, 'preprocessing_telemetry', DATA / 'processed/supervised')[0]]
            partes.extend(tabular_data([r], 'preprocessing_cassette', Path(gettempdir()) / 'sleep-edfx-sc-features-v1')[0] for r in records_sc)
            tabla = pd.concat(partes, ignore_index=True)
        x_train, y_train = tabla.loc[train], meta.loc[train, 'stage']
        modelo = LGBMClassifier(**dict(spec['parameters'], n_jobs=THREADS))
        inicio = time.perf_counter()
        modelo.fit(x_train, y_train)
        info['ajuste_dart_s'] = time.perf_counter() - inicio
        modelo.booster_.save_model(str(rutas['dart']))
        info['clases_dart'] = modelo.classes_.tolist()
        guardar()
        del tabla, x_train, modelo
    original = CACHE / 'v6-1'
    if not (original / 'encoder.weights.h5').exists():
        raise FileNotFoundError('Falta el extractor entrenado en la primera ronda.')
    encoder = train_encoder(train, 1, original)
    if not rutas['v6'].exists() or 'ajuste_cabezal_hmm_s' not in info:
        feats = np.empty((len(train), 192), dtype='float32')
        for rid in meta.loc[train, 'record_id'].unique():
            mask = meta.loc[train, 'record_id'].to_numpy() == rid
            feats[mask] = np.load(original / f'{rid}-192.npy', mmap_mode='r')[epoch_number[train[mask]]]
        inicio = time.perf_counter()
        head = LGBMClassifier(**hybrid_params).fit(feats, y_num[train])
        fit_hmm(meta.loc[train])
        info['ajuste_cabezal_hmm_s'] = time.perf_counter() - inicio
        head.booster_.save_model(str(rutas['v6']))
        guardar()
        del feats, head
    trans, prior = fit_hmm(meta.loc[train])
    boosters = {k: Booster(model_file=str(path)) for k, path in rutas.items()}
    # Calentamiento de cada modelo antes de cronometrar. No se usa la etiqueta para predecir.
    for b in boosters.values():
        b.predict(np.zeros((1, b.num_feature())), num_threads=THREADS)
    contextual_embeddings(encoder, np.zeros((BATCH,3000,4), dtype='float32'), batch_size=BATCH)
    noches = test.groupby('cohort').record_id.min()
    info['dispositivo_crnn'] = 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'
    info['hilos_lightgbm'] = THREADS
    for cohorte, rid in noches.items():
        registro = lookup[rid]
        for modelo in ('dart', 'v6'):
            if any(m['registro']==rid and m['modelo']==modelo for m in info['inferencia']):
                continue
            inicio = time.perf_counter()
            if modelo == 'dart':
                perfil = spec['preprocessing_telemetry' if cohorte == 'Telemetry' else 'preprocessing_cassette']
                entrada = PreprocessingPipeline(PreprocessingConfig(**perfil)).transform_edf(registro.psg_path)
                entrada = entrada.features.astype('float32').reindex(columns=spec['feature_columns'])
                listo = time.perf_counter()
                pred = np.array(info['clases_dart'])[boosters[modelo].predict(entrada, num_threads=THREADS).argmax(axis=1)]
            else:
                raw = mne.io.read_raw_edf(registro.psg_path, preload=True, verbose='ERROR')
                presentes = [ch for ch in channels if ch in raw.ch_names]
                raw.pick(presentes).filter(0.3, 35.0, verbose=False)
                n = int(raw.n_times // 3000)
                entrada = np.zeros((n, 3000, len(channels)), dtype='float32')
                for j, ch in enumerate(channels):
                    if ch in presentes:
                        v = raw.get_data(picks=[ch], stop=n*3000).reshape(n,3000) * 1e6
                        entrada[:,:,j] = (v-v.mean(1,keepdims=True))/(v.std(1,keepdims=True)+1e-8)
                listo = time.perf_counter()
                z = contextual_embeddings(encoder, entrada, batch_size=BATCH)
                pred = np.array(STAGES)[decode_night(boosters[modelo].predict(z, num_threads=THREADS), trans, prior)]
            fin = time.perf_counter()
            previo = pd.read_parquet(CACHE / f'fold-1-{modelo}.parquet').query('record_id == @rid')
            assert len(previo) and len(pred) > previo.epoch_index.max()
            info['inferencia'].append(dict(modelo=modelo, registro=rid, cohorte=cohorte,
                segmentos_edf=len(pred), preparacion_s=listo-inicio, prediccion_s=fin-listo,
                total_s=fin-inicio, coincidencia=float(np.mean(pred[previo.epoch_index] == previo.prediction))))
            guardar()
            print(rid, modelo, round(fin-inicio, 2), 's', flush=True)
    return info


# La medición guardada se muestra en la sección 5. Para completar una medici?n pendiente con el kernel
# del proyecto, ejecutar medir_tiempos_ronda_1() tras preparar señales y datos.


## 5. Resultados fuera de muestra

**Estado del V6 (18 de noviembre).** Entrenamiento en pausa. La primera ronda terminó y se conserva el punto de recuperación de la época 11 de 20 en la segunda ronda. El experimento prioritario DART + CRNN pequeña ya completó sus cinco rondas en `lightgbm_transiciones_sueno.ipynb`.

### Qué modelo va mejor

Pruebas guardadas de cinco rondas por persona: LightGBM **5/5** y V6 **1/5**.

V6 usa la CRNN con ventanas de 15 segmentos, LightGBM con 192 características temporales y decodificación HMM por noche (fase B, una pasada).

**Resultado provisional.** En las rondas compartidas (1), LightGBM mixto obtiene el mayor F1 macro, con una ventaja de 0,0445.

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |

### Rendimiento por ronda y cohorte de prueba

| Ronda | Cohorte | Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Ambas | LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| 1 | Cassette | LightGBM mixto | 3 | 3 | 8039 | 0,7394 | 91,67 % | 0,8296 |
| 1 | Telemetry | LightGBM mixto | 4 | 8 | 7854 | 0,8262 | 83,13 % | 0,7724 |
| 1 | Ambas | CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |
| 1 | Cassette | CRNN + LightGBM 192D + HMM | 3 | 3 | 8039 | 0,6337 | 80,98 % | 0,6488 |
| 1 | Telemetry | CRNN + LightGBM 192D + HMM | 4 | 8 | 7854 | 0,7985 | 83,69 % | 0,7731 |

### Matrices de confusión de las rondas compartidas

Cada fila indica la etapa real y cada columna la predicción. Los valores cuentan segmentos de 30 segundos.

**LightGBM mixto**

| Real / predicha | W | N1 | N2 | N3 | REM |
| --- | --- | --- | --- | --- | --- |
| W | 6147 | 102 | 13 | 4 | 11 |
| N1 | 102 | 768 | 183 | 4 | 140 |
| N2 | 68 | 306 | 3926 | 570 | 253 |
| N3 | 2 | 0 | 42 | 1168 | 0 |
| REM | 4 | 37 | 150 | 4 | 1889 |

**CRNN + LightGBM 192D + HMM**

| Real / predicha | W | N1 | N2 | N3 | REM |
| --- | --- | --- | --- | --- | --- |
| W | 5158 | 193 | 887 | 4 | 35 |
| N1 | 101 | 582 | 230 | 2 | 282 |
| N2 | 7 | 177 | 4466 | 322 | 151 |
| N3 | 1 | 1 | 185 | 1025 | 0 |
| REM | 2 | 18 | 210 | 2 | 1852 |

### Rendimiento global sobre las cinco rondas

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 34 | 56 | 75612 | 0,8010 | 85,96 % | 0,8074 |

### Datos de entrenamiento y prueba

Los dos modelos comparten esta partición. Cada noche se cuenta una vez en prueba y cuatro veces en entrenamiento a lo largo de la validación cruzada. Una persona conserva todas sus noches en la misma ronda de prueba.

| Cohorte | Personas disponibles | Noches disponibles | Segmentos etiquetados |
| --- | --- | --- | --- |
| Cassette | 12 | 12 | 32921 |
| Telemetry | 22 | 44 | 42691 |

| Ronda | Cohorte | Entrenamiento · personas | Noches | Segmentos | Prueba · personas | Noches | Segmentos |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Cassette | 9 | 9 | 24882 | 3 | 3 | 8039 |
| 1 | Telemetry | 18 | 36 | 34837 | 4 | 8 | 7854 |
| 2 | Cassette | 9 | 9 | 24681 | 3 | 3 | 8240 |
| 2 | Telemetry | 18 | 36 | 35168 | 4 | 8 | 7523 |
| 3 | Cassette | 10 | 10 | 27327 | 2 | 2 | 5594 |
| 3 | Telemetry | 17 | 34 | 32876 | 5 | 10 | 9815 |
| 4 | Cassette | 10 | 10 | 27431 | 2 | 2 | 5490 |
| 4 | Telemetry | 17 | 34 | 33192 | 5 | 10 | 9499 |
| 5 | Cassette | 10 | 10 | 27363 | 2 | 2 | 5558 |
| 5 | Telemetry | 18 | 36 | 34691 | 4 | 8 | 8000 |

### Qué noches quedaron en prueba

Para reconstruir el entrenamiento de una ronda, toma las noches de las otras cuatro rondas de esta tabla.

| Ronda | Cohorte | Identificadores de los registros de prueba |
| --- | --- | --- |
| 1 | Cassette | SC4101, SC4342, SC4352 |
| 1 | Telemetry | ST7051, ST7052, ST7141, ST7142, ST7211, ST7212, ST7221, ST7222 |
| 2 | Cassette | SC4262, SC4522, SC4722 |
| 2 | Telemetry | ST7071, ST7072, ST7151, ST7152, ST7191, ST7192, ST7241, ST7242 |
| 3 | Cassette | SC4572, SC4821 |
| 3 | Telemetry | ST7041, ST7042, ST7091, ST7092, ST7101, ST7102, ST7121, ST7122, ST7201, ST7202 |
| 4 | Cassette | SC4072, SC4172 |
| 4 | Telemetry | ST7021, ST7022, ST7061, ST7062, ST7111, ST7112, ST7131, ST7132, ST7171, ST7172 |
| 5 | Cassette | SC4062, SC4411 |
| 5 | Telemetry | ST7011, ST7012, ST7081, ST7082, ST7161, ST7162, ST7181, ST7182 |

### Tiempo de entrenamiento de la primera ronda

| Modelo / componente | Qué se cronometró | Segundos | Duración |
| --- | --- | --- | --- |
| LightGBM mixto | Ajuste repetido sobre las características del mismo entrenamiento (CPU) | 420,7 | 0 h 07 min 01 s |
| V6 · extractor CRNN | Suma de 20 épocas del registro original de Keras (RTX 3080) | 18042,0 | 5 h 00 min 42 s |
| V6 · extracción de vectores | Estimación para 45 noches de entrenamiento; intervalos entre guardados consecutivos de la ejecución original | 966,7 | 0 h 16 min 07 s |
| V6 · LightGBM + HMM | Reajuste sobre los vectores guardados de la primera ronda (CPU) | 25,5 | 0 h 00 min 26 s |
| V6 · total aproximado | Suma de las tres etapas anteriores sobre el entrenamiento | 19034,2 | 5 h 17 min 14 s |

Las mediciones corresponden a la primera ronda, con las entradas preparadas. El tiempo de ajuste de LightGBM comienza con las características tabulares disponibles. En V6 comienza con las señales filtradas disponibles. El total V6 combina registros originales con un nuevo cronometraje del clasificador final.

### Tiempo de inferencia de una noche completa desde el EDF

Una pasada por modelo sobre la primera noche de prueba de cada cohorte, ordenada por identificador. Modelos en memoria, tras calentamiento. CRNN en GPU y LightGBM en CPU. El total incluye lectura, preprocesamiento y predicción de todos los segmentos del EDF. El F1 usa solo los segmentos con etiqueta válida.

| Cohorte | Noche | Modelo | Segmentos EDF | Preparación (s) | Predicción (s) | Total (s) | Coincidencia con predicción guardada |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Cassette | SC4101 | LightGBM mixto | 2720 | 30,05 | 0,10 | 30,15 | 100,00 % |
| Cassette | SC4101 | CRNN + LightGBM 192D + HMM | 2720 | 2,21 | 43,27 | 45,48 | 100,00 % |
| Telemetry | ST7051 | LightGBM mixto | 1073 | 15,86 | 0,04 | 15,89 | 99,80 % |
| Telemetry | ST7051 | CRNN + LightGBM 192D + HMM | 1073 | 0,56 | 16,63 | 17,18 | 100,00 % |

Al repetir LightGBM en ST7051 desde el EDF, 2 de 1018 segmentos etiquetados cambiaron de predicción. Con el mismo clasificador y las características tabulares guardadas se reprodujeron las 1018 predicciones. Las tablas de rendimiento usan las predicciones originales de prueba.


In [2]:
faltantes = [f'{m} · ronda {f}' for m in ('dart','v6') for f in range(1,6)
             if not (CACHE / f'fold-{f}-{m}.parquet').is_file()]
if faltantes:
    print('Análisis global de ambos modelos pendiente de: ' + ', '.join(faltantes))
else:
    keys = ['record_id', 'subject', 'epoch_index', 'stage', 'fold']
    oof = meta.copy()
    for model in ['dart', 'v6']:
        predictions = pd.concat([pd.read_parquet(CACHE / f'fold-{i}-{model}.parquet') for i in range(1, FOLDS + 1)])
        assert len(predictions) == len(meta) and predictions.prediction.isin(STAGES).all()
        oof = oof.merge(predictions.rename(columns={'prediction': model}), on=keys, validate='one_to_one', how='left')
        assert oof[model].notna().all()
    assert len(oof) == len(meta)
    oof.to_parquet(CACHE / 'predicciones_fuera_de_muestra.parquet', index=False)

    def metrics(frame):
        rows = []
        for name in ['dart', 'v6']:
            y, pred = frame.stage, frame[name]
            rows.append({'modelo': name, 'sujetos': frame.subject.nunique(), 'épocas': len(y),
                         'F1 macro': f1_score(y, pred, labels=STAGES, average='macro', zero_division=0),
                         'exactitud': accuracy_score(y, pred), 'kappa': cohen_kappa_score(y, pred),
                         'F1 N1': f1_score(y, pred, labels=['N1'], average=None, zero_division=0)[0]})
        return pd.DataFrame(rows).set_index('modelo')

    display(pd.concat({'Completa': metrics(oof), 'Recorte ±30 min': metrics(oof[oof.trim])}).round(4))
    for group in ['cohort', 'fold', 'subject']:
        display(pd.concat({key: metrics(frame) for key, frame in oof.groupby(group)}).round(4))
    for model in ['dart', 'v6']:
        print(model, '· filas reales, columnas predichas')
        display(pd.DataFrame(confusion_matrix(oof.stage, oof[model], labels=STAGES), index=STAGES, columns=STAGES))

Análisis global de ambos modelos pendiente de: v6 · ronda 2, v6 · ronda 3, v6 · ronda 4, v6 · ronda 5


In [3]:
# Reutiliza solo la celda de lectura del informe si se abre un kernel nuevo.
if 'mostrar_resultados' not in globals():
    import json
    from pathlib import Path
    raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.dvc/config').is_file())
    archivo = raiz / 'Notebooks/experiments/comparacion_justa_lightgbm_v6.ipynb'
    celda = next(c for c in json.loads(archivo.read_text('utf-8'))['cells'] if c['id'] == 'cv-resultados')
    exec(''.join(celda['source']), globals())

reporte = mostrar_resultados(CACHE)
print('Reporte de prueba calculado con las predicciones guardadas.')
display(Markdown(reporte))

Reporte de prueba calculado con las predicciones guardadas.


### Qué modelo va mejor

Pruebas guardadas de cinco rondas por persona: LightGBM **5/5** y V6 **1/5**.

V6 usa la CRNN con ventanas de 15 segmentos, LightGBM con 192 características temporales y decodificación HMM por noche (fase B, una pasada).

**Resultado provisional.** En las rondas compartidas (1), LightGBM mixto obtiene el mayor F1 macro, con una ventaja de 0,0445.

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |

### Rendimiento por ronda y cohorte de prueba

| Ronda | Cohorte | Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Ambas | LightGBM mixto | 7 | 11 | 15893 | 0,8194 | 87,45 % | 0,8262 |
| 1 | Cassette | LightGBM mixto | 3 | 3 | 8039 | 0,7394 | 91,67 % | 0,8296 |
| 1 | Telemetry | LightGBM mixto | 4 | 8 | 7854 | 0,8262 | 83,13 % | 0,7724 |
| 1 | Ambas | CRNN + LightGBM 192D + HMM | 7 | 11 | 15893 | 0,7749 | 82,32 % | 0,7536 |
| 1 | Cassette | CRNN + LightGBM 192D + HMM | 3 | 3 | 8039 | 0,6337 | 80,98 % | 0,6488 |
| 1 | Telemetry | CRNN + LightGBM 192D + HMM | 4 | 8 | 7854 | 0,7985 | 83,69 % | 0,7731 |

### Matrices de confusión de las rondas compartidas

Cada fila indica la etapa real y cada columna la predicción. Los valores cuentan segmentos de 30 segundos.

**LightGBM mixto**

| Real / predicha | W | N1 | N2 | N3 | REM |
| --- | --- | --- | --- | --- | --- |
| W | 6147 | 102 | 13 | 4 | 11 |
| N1 | 102 | 768 | 183 | 4 | 140 |
| N2 | 68 | 306 | 3926 | 570 | 253 |
| N3 | 2 | 0 | 42 | 1168 | 0 |
| REM | 4 | 37 | 150 | 4 | 1889 |

**CRNN + LightGBM 192D + HMM**

| Real / predicha | W | N1 | N2 | N3 | REM |
| --- | --- | --- | --- | --- | --- |
| W | 5158 | 193 | 887 | 4 | 35 |
| N1 | 101 | 582 | 230 | 2 | 282 |
| N2 | 7 | 177 | 4466 | 322 | 151 |
| N3 | 1 | 1 | 185 | 1025 | 0 |
| REM | 2 | 18 | 210 | 2 | 1852 |

### Rendimiento global sobre las cinco rondas

| Modelo | Personas | Noches | Segmentos de 30 s | F1 macro | Exactitud | Kappa |
| --- | --- | --- | --- | --- | --- | --- |
| LightGBM mixto | 34 | 56 | 75612 | 0,8010 | 85,96 % | 0,8074 |

### Datos de entrenamiento y prueba

Los dos modelos comparten esta partición. Cada noche se cuenta una vez en prueba y cuatro veces en entrenamiento a lo largo de la validación cruzada. Una persona conserva todas sus noches en la misma ronda de prueba.

| Cohorte | Personas disponibles | Noches disponibles | Segmentos etiquetados |
| --- | --- | --- | --- |
| Cassette | 12 | 12 | 32921 |
| Telemetry | 22 | 44 | 42691 |

| Ronda | Cohorte | Entrenamiento · personas | Noches | Segmentos | Prueba · personas | Noches | Segmentos |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Cassette | 9 | 9 | 24882 | 3 | 3 | 8039 |
| 1 | Telemetry | 18 | 36 | 34837 | 4 | 8 | 7854 |
| 2 | Cassette | 9 | 9 | 24681 | 3 | 3 | 8240 |
| 2 | Telemetry | 18 | 36 | 35168 | 4 | 8 | 7523 |
| 3 | Cassette | 10 | 10 | 27327 | 2 | 2 | 5594 |
| 3 | Telemetry | 17 | 34 | 32876 | 5 | 10 | 9815 |
| 4 | Cassette | 10 | 10 | 27431 | 2 | 2 | 5490 |
| 4 | Telemetry | 17 | 34 | 33192 | 5 | 10 | 9499 |
| 5 | Cassette | 10 | 10 | 27363 | 2 | 2 | 5558 |
| 5 | Telemetry | 18 | 36 | 34691 | 4 | 8 | 8000 |

### Qué noches quedaron en prueba

Para reconstruir el entrenamiento de una ronda, toma las noches de las otras cuatro rondas de esta tabla.

| Ronda | Cohorte | Identificadores de los registros de prueba |
| --- | --- | --- |
| 1 | Cassette | SC4101, SC4342, SC4352 |
| 1 | Telemetry | ST7051, ST7052, ST7141, ST7142, ST7211, ST7212, ST7221, ST7222 |
| 2 | Cassette | SC4262, SC4522, SC4722 |
| 2 | Telemetry | ST7071, ST7072, ST7151, ST7152, ST7191, ST7192, ST7241, ST7242 |
| 3 | Cassette | SC4572, SC4821 |
| 3 | Telemetry | ST7041, ST7042, ST7091, ST7092, ST7101, ST7102, ST7121, ST7122, ST7201, ST7202 |
| 4 | Cassette | SC4072, SC4172 |
| 4 | Telemetry | ST7021, ST7022, ST7061, ST7062, ST7111, ST7112, ST7131, ST7132, ST7171, ST7172 |
| 5 | Cassette | SC4062, SC4411 |
| 5 | Telemetry | ST7011, ST7012, ST7081, ST7082, ST7161, ST7162, ST7181, ST7182 |

### Tiempo de entrenamiento de la primera ronda

| Modelo / componente | Qué se cronometró | Segundos | Duración |
| --- | --- | --- | --- |
| LightGBM mixto | Ajuste repetido sobre las características del mismo entrenamiento (CPU) | 420,7 | 0 h 07 min 01 s |
| V6 · extractor CRNN | Suma de 20 épocas del registro original de Keras (RTX 3080) | 18042,0 | 5 h 00 min 42 s |
| V6 · extracción de vectores | Estimación para 45 noches de entrenamiento; intervalos entre guardados consecutivos de la ejecución original | 966,7 | 0 h 16 min 07 s |
| V6 · LightGBM + HMM | Reajuste sobre los vectores guardados de la primera ronda (CPU) | 25,5 | 0 h 00 min 26 s |
| V6 · total aproximado | Suma de las tres etapas anteriores sobre el entrenamiento | 19034,2 | 5 h 17 min 14 s |

Las mediciones corresponden a la primera ronda, con las entradas preparadas. El tiempo de ajuste de LightGBM comienza con las características tabulares disponibles. En V6 comienza con las señales filtradas disponibles. El total V6 combina registros originales con un nuevo cronometraje del clasificador final.

### Tiempo de inferencia de una noche completa desde el EDF

Una pasada por modelo sobre la primera noche de prueba de cada cohorte, ordenada por identificador. Modelos en memoria, tras calentamiento. CRNN en GPU y LightGBM en CPU. El total incluye lectura, preprocesamiento y predicción de todos los segmentos del EDF. El F1 usa solo los segmentos con etiqueta válida.

| Cohorte | Noche | Modelo | Segmentos EDF | Preparación (s) | Predicción (s) | Total (s) | Coincidencia con predicción guardada |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Cassette | SC4101 | LightGBM mixto | 2720 | 30,05 | 0,10 | 30,15 | 100,00 % |
| Cassette | SC4101 | CRNN + LightGBM 192D + HMM | 2720 | 2,21 | 43,27 | 45,48 | 100,00 % |
| Telemetry | ST7051 | LightGBM mixto | 1073 | 15,86 | 0,04 | 15,89 | 99,80 % |
| Telemetry | ST7051 | CRNN + LightGBM 192D + HMM | 1073 | 0,56 | 16,63 | 17,18 | 100,00 % |

Al repetir LightGBM en ST7051 desde el EDF, 2 de 1018 segmentos etiquetados cambiaron de predicción. Con el mismo clasificador y las características tabulares guardadas se reprodujeron las 1018 predicciones. Las tablas de rendimiento usan las predicciones originales de prueba.
